# distributed-sampler-shard — faded example 2: Collect shards across multiple epochs with set_epoch called each time

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `distributed-sampler-shard`. The last cell reports your progress on the `Distributed: DistributedSampler shard` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: DistributedSampler shard` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`distributed-sampler-shard`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "distributed-sampler-shard"
DD_SUBTOPIC = "Distributed: DistributedSampler shard"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

To inspect how `DistributedSampler` changes the data order across epochs, you build the sampler fresh (or reuse with `set_epoch`) for each `(epoch, rank)` combination. The critical discipline is calling `set_epoch(epoch)` before each iteration — it seeds the shuffle so rank allocations differ across epochs while remaining balanced within each epoch.

## Faded exercise 2

Complete `collect_multi_epoch`. The outer loop is provided. Fill in the inner construction, `set_epoch` call, and index collection.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
from torch.utils.data.distributed import DistributedSampler

def collect_multi_epoch(dataset, world_size, seed, num_epochs):
    result = []  # result[epoch][rank] = list of indices
    for epoch in range(num_epochs):
        epoch_row = []
        for rank in range(world_size):
            raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
        result.append(epoch_row)
    return result


def _test():
    import torch
    from torch.utils.data import TensorDataset
    from itertools import chain

    n = 12
    world_size = 3
    dataset = TensorDataset(torch.arange(n))
    seed = 7
    num_epochs = 4

    result = collect_multi_epoch(dataset, world_size, seed, num_epochs)

    assert len(result) == num_epochs
    assert all(len(row) == world_size for row in result)

    # Each epoch: union covers full dataset
    for epoch in range(num_epochs):
        assert set(chain(*result[epoch])) == set(range(n))

    # Different epochs produce different permutations for rank 0
    epoch0_rank0 = result[0][0]
    different_count = sum(result[e][0] != epoch0_rank0 for e in range(1, num_epochs))
    assert different_count > 0, 'set_epoch is not changing the permutation'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from torch.utils.data.distributed import DistributedSampler

def collect_multi_epoch(dataset, world_size, seed, num_epochs):
    result = []  # result[epoch][rank] = list of indices
    for epoch in range(num_epochs):
        epoch_row = []
        for rank in range(world_size):
            sampler = DistributedSampler(
                dataset, num_replicas=world_size, rank=rank,
                shuffle=True, seed=seed
            )
            sampler.set_epoch(epoch)
            epoch_row.append(list(sampler))
        result.append(epoch_row)
    return result
```
</details>